# 05 — Diagnose the raw content of `mit/oceQsw`

Notebook 04 produced a July-2020 model mean of **−19.9 W m⁻²** for `oceQsw`
(ocean, 60°S–60°N). A monthly-mean net downward shortwave flux is strictly positive
(CERES: +165 W m⁻²), and a pure sign-convention flip would give −165, not −20 — so
the values being read are **not net shortwave** under any sign convention. Candidate
explanations:

1. the directory actually holds a different field (e.g., net total heat flux in the
   MITgcm forcing convention, positive up — a July ocean mean of ≈ −20 W m⁻² fits that);
2. a few corrupt/placeholder files in the staging area poison the monthly mean;
3. a reader problem (dtype, endianness, layout) — unlikely, since the field is smooth.

The two cheap tests below separate these unambiguously. Shortwave has an unmistakable
fingerprint: **exactly zero all night**, peaks of +800–1000 W m⁻² at local noon
(negative if stored positive-up), and an instantaneous snapshot shows one bright day
hemisphere. A Qnet-type field instead goes **negative at night** (longwave + turbulent
cooling) and positive by day.

No dask cluster needed; the probe-point series reads 4 bytes per file.

In [ ]:
# Environment check: run on SciServer (Kraken domain, with the Poseidon DYAMOND
# ceph volume attached), or set DYAMOND_ROOT to a local subset.
# SciServer containers do not persist `pip install --user` across restarts, so
# fall back to importing directly from the repo's src/ tree if needed.
try:
    from dyamond_fluxes import dyamond_root
except ModuleNotFoundError:
    import sys
    from pathlib import Path as _P

    sys.path.insert(0, str((_P.cwd() / ".." / "src").resolve()))
    from dyamond_fluxes import dyamond_root

root = dyamond_root()
print(f"DYAMOND root: {root}")

## 1. What does the staging `readme.txt` say?

If the mit/ tree documents its variables and conventions, that may settle it directly.

In [ ]:
from dyamond_fluxes import mds

readme = mds.mit_dir() / "readme.txt"
print(readme.read_text() if readme.exists() else "-- no readme.txt found --")

## 2. Probe point: hourly July time series at one equatorial Pacific cell

Pick a deep open-ocean cell far from land, ice, and the Equatorial Front, and read
that single value from every hourly file (a 4-byte seek per file — seconds, not
minutes). `oceQnet` is read alongside for comparison: if the two series coincide,
the `oceQsw` directory is mislabeled.

In [ ]:
import numpy as np

from dyamond_fluxes import open_grid

TARGET_LON, TARGET_LAT = -140.0, 0.0  # equatorial Pacific

grid = open_grid()
dist = (grid.XC - TARGET_LON) ** 2 + (grid.YC - TARGET_LAT) ** 2
flat = int(np.nanargmin(dist.where(grid.Depth > 1000).values))
face, j, i = np.unravel_index(flat, grid.XC.shape)
print(
    f"probe: face={face} j={j} i={i}  "
    f"lon={float(grid.XC.values[face, j, i]):.2f}  "
    f"lat={float(grid.YC.values[face, j, i]):.2f}  "
    f"depth={float(grid.Depth.values[face, j, i]):.0f} m"
)

In [ ]:
N = 2160
NPTS = 13 * N * N


def point_series(var: str, start: str, end: str):
    """Value at the probe cell from every file of `var` in [start, end)."""
    vdir = mds.mit_dir() / var
    files = sorted(vdir.glob(f"{var}.*.data"))
    if not files:
        raise FileNotFoundError(f"no {var}.*.data under {vdir}")
    iters = np.array([int(f.name.split(".")[1]) for f in files])
    times = mds.iters_to_time(iters)
    print(f"{var}: {len(files)} files total, {times[0]} to {times[-1]}")

    keep = (times >= np.datetime64(start)) & (times < np.datetime64(end))
    files = [f for f, k in zip(files, keep) if k]
    times = times[keep]

    itemsize = files[0].stat().st_size // NPTS
    dtype = {4: ">f4", 8: ">f8"}[itemsize]
    vals = np.empty(len(files))
    for m, f in enumerate(files):
        with open(f, "rb") as fh:
            fh.seek(flat * itemsize)
            vals[m] = np.frombuffer(fh.read(itemsize), dtype=dtype)[0]
    return times, vals


series = {v: point_series(v, "2020-07-01", "2020-08-01") for v in ("oceQsw", "oceQnet")}
for v, (t, vals) in series.items():
    print(
        f"{v:8s} July @probe: n={vals.size:4d}  min={vals.min():8.1f}  "
        f"max={vals.max():8.1f}  mean={vals.mean():7.1f}  "
        f"exactly zero: {(vals == 0).mean():5.1%}"
    )

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
for v, (t, vals) in series.items():
    axes[0].plot(t[: 5 * 24], vals[: 5 * 24], marker=".", ms=3, label=v)  # first 5 days
    axes[1].plot(t, vals, lw=0.5, label=v)                                # full month
axes[0].set_title("Probe cell, first 5 days of July 2020 (hourly)")
axes[1].set_title("Probe cell, full July 2020 — any corrupt file shows as a spike")
for ax in axes:
    ax.axhline(0, color="k", lw=0.5)
    ax.set_ylabel("raw value")
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()

## 3. One instantaneous snapshot: statistics and day/night geography

At 2020-07-15 12:00 UTC the subsolar point is near (0°E, 21°N): if this is shortwave
(positive down), the map shows a single bright patch centered there and zeros over the
night hemisphere (Pacific).

In [ ]:
from dyamond_fluxes import bin_to_latlon, open_mds_variable

snap = open_mds_variable("oceQsw").sel(
    time=np.datetime64("2020-07-15T12:00:00"), method="nearest"
)
print("snapshot time:", snap.time.values)
field = snap.load().where(grid.Depth > 0)

vals = field.values[np.isfinite(field.values)]
qs = np.percentile(vals, [0, 0.1, 1, 50, 99, 99.9, 100])
print("percentiles [min, 0.1, 1, 50, 99, 99.9, max]:", np.round(qs, 1))
print(f"ocean cells negative: {(vals < 0).mean():5.1%}   exactly zero: {(vals == 0).mean():5.1%}")

binned = bin_to_latlon(field, grid.XC, grid.YC, area=grid.rA)
fig, ax = plt.subplots(figsize=(11, 4.5))
pc = ax.pcolormesh(binned.lon, binned.lat, binned, cmap="RdBu_r", vmin=-1000, vmax=1000)
fig.colorbar(pc, ax=ax, label="raw value")
ax.set_title(f"raw oceQsw snapshot, {snap.time.values} (red = positive)")

## Interpretation

| Probe-series signature | Meaning | Fix |
| --- | --- | --- |
| Flat **zero every night**, day peaks **+800 to +1000** | It *is* SW, positive down — the July mean was poisoned elsewhere. Look for spikes in the full-month panel (corrupt staging files) and delete the stale cache (`rm qsw_model_2020-07_1deg.nc`) before rerunning 04. | Exclude bad iterations; recompute. |
| Flat zero every night, day peaks **−800 to −1000** | SW stored positive **up**. (Alone this predicts a −165 mean, so check for spikes too.) | Set the convention in `mds.MITGCM_DIAG_ATTRS`; flip. |
| **Negative at night** (≈ −200), positive by day; `oceQsw` ≈ `oceQnet` | Directory mislabeled / holds a net-heat-flux-type field. | Re-map variables per `readme.txt`. |
| Values ~O(1e±30) noise | Reader dtype/layout wrong (unlikely — the binned field was smooth). | Fix `mds._detect_layout`. |

Paste the `readme.txt` text, the printed statistics, and (if convenient) the two figures
back into the chat and we will patch the reader/notebooks accordingly. Whatever the
outcome, **delete `qsw_model_2020-07_1deg.nc`** before rerunning notebook 04 — it caches
the bad mean.